In [2]:
pip install transformers torch gradio langchain langchain-community ollama langchain-ollama

INFO: pip is looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
  Using cached gradio-6.20.0-py3-none-any.whl.metadata (17 kB)
  Using cached starlette-1.3.1-py3-none-any.whl.metadata (6.4 kB)
INFO: pip is looking at multiple versions of fastapi to determine which version is compatible with other requirements. This could take a while.
  Using cached fastapi-0.139.2-py3-none-any.whl.metadata (26 kB)
   ---------------------------------------- 0.0/32.3 MB ? eta -:--:--
   - -------------------------------------- 1.0/32.3 MB 9.0 MB/s eta 0:00:04
   --- ------------------------------------ 3.1/32.3 MB 9.2 MB/s eta 0:00:04
   ------- -------------------------------- 6.3/32.3 MB 11.4 MB/s eta 0:00:03
   ------------ --------------------------- 9.7/32.3 MB 12.6 MB/s eta 0:00:02
   --------------- ------------------------ 12.6/32.3 MB 13.7 MB/s eta 0:00:02
   -------------------- ------------------- 16.5/32.3 MB 13.

In [2]:
pip install ffmpeg

  Using cached ffmpeg-1.4.tar.gz (5.1 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for ffmpeg: filename=ffmpeg-1.4-py3-none-any.whl size=6136 sha256=bbf8e3104d0044325c61bbde1cf2a9e49990259370aa4bca6800a698006db3b0
  Stored in directory: c:\users\ameta\appdata\local\pip\cache\wheels\a4\04\6c\ab972358c48aedc5be02f1d28968f2baa491a2837270932043
Successfully built ffmpeg
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/hTqGqoC-LrW6S79HjuJUkg/trimmed-02.wav"
res = requests.get(url)

audio_file_path = "sample-meeting.wav"

if res.status_code == 200:
	# If successful, write the content to the specified local file path
	with open(audio_file_path, "wb") as file:
		file.write(res.content)
		print("File downloaded successfully")
else:
	# If the request failed, print an error message
	print("Failed to download the file")
 
#Implement openai whisper for voice to speech 

import torch 
from transformers import pipeline

pipe = pipeline(
  "automatic-speech-recognition",
  model="openai/whisper-tiny.en",
  chunk_length_s=30,
)
sample = 'sample-meeting.wav'
prediction = pipe(
    sample,
    batch_size=8
)
["text"]
print(prediction)

File downloaded successfully


In [1]:
import gradio as gr

def greet(name):
	return "Hello " + name + "!"

demo = gr.Interface(fn=greet, inputs="text", outputs="text")

demo.launch(server_name="0.0.0.0", server_port= 5000)


* Running on local URL:  http://0.0.0.0:5000
* To create a public link, set `share=True` in `launch()`.


C:\Users\ameta\anaconda3\Lib\site-packages\gradio\routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [10]:
pip install torchaudio

Note: you may need to restart the kernel to use updated packages.


In [14]:
import torch
from transformers import pipeline
import gradio as gr
import numpy as np

# Pipeline ek baar hi initialize karo (function ke bahar) — 
# har request pe model reload karna bahut slow hai
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny.en",
    chunk_length_s=30,
)

TARGET_SR = 16000  # Whisper 16kHz expect karta hai

def transcript_audio(audio_file):
    # type="numpy" se Gradio khud (sampling_rate, numpy_array) tuple deta hai —
    # FFmpeg ki zaroorat hi nahi padti, filepath decode karne ki
    sampling_rate, audio_array = audio_file

    # Stereo ko mono mein convert karo (Whisper sirf 1D array leta hai)
    if audio_array.ndim > 1:
        audio_array = audio_array.mean(axis=1)

    # Whisper float32 audio expect karta hai, normalize kar rahe hain
    audio_array = audio_array.astype(np.float32)
    max_val = np.abs(audio_array).max()
    if max_val > 0:
        audio_array = audio_array / max_val

    result = pipe(
        {"array": audio_array, "sampling_rate": sampling_rate},
        batch_size=8
    )["text"]
    return result

# Gradio interface
audio_input = gr.Audio(sources="upload", type="numpy")   # filepath ki jagah numpy
output_text = gr.Textbox()

iface = gr.Interface(
    fn=transcript_audio,
    inputs=audio_input,
    outputs=output_text,
    title="Audio Transcription App",
    description="Upload the audio file"
)

iface.launch(server_name="0.0.0.0", server_port=9000)

Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


* Running on local URL:  http://0.0.0.0:9000
* To create a public link, set `share=True` in `launch()`.


C:\Users\ameta\anaconda3\Lib\site-packages\gradio\routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Traceback (most recent call last):
  File "C:\Users\ameta\anaconda3\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "C:\Users\ameta\anaconda3\Lib\site-packages\gradio\route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "C:\Users\ameta\anaconda3\Lib\site-packages\gradio\blocks.py", line 2277, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "C:\Users\ameta\anaconda3\Lib\site-p

In [16]:
import numpy as np
import gradio as gr
from transformers import pipeline

# Load the Whisper model once at startup (not inside the function)
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny.en",
    chunk_length_s=30,
)


def transcribe_audio(audio):
    """
    audio: tuple (sampling_rate, numpy_array) — provided by Gradio when
    type="numpy" is used, so no file decoding (and no FFmpeg) is needed.
    """
    if audio is None:
        return "No audio provided."

    sampling_rate, audio_array = audio

    # Convert to mono if stereo (average the channels)
    if audio_array.ndim > 1:
        audio_array = audio_array.mean(axis=1)

    # Normalize to float32 in range [-1, 1] — Whisper expects this
    audio_array = audio_array.astype(np.float32)
    max_val = np.abs(audio_array).max()
    if max_val > 0:
        audio_array = audio_array / max_val

    result = pipe({"array": audio_array, "sampling_rate": sampling_rate})
    return result["text"]


# Gradio interface — supports both file upload and microphone recording
audio_input = gr.Audio(sources=["upload", "microphone"], type="numpy", label="Speak or Upload Audio")
output_text = gr.Textbox(label="Transcription")

iface = gr.Interface(
    fn=transcribe_audio,
    inputs=audio_input,
    outputs=output_text,
    title="Voice to Text Generator",
    description="Upload an audio file or record from your microphone to get the transcription.",
)

if __name__ == "__main__":
    iface.launch(server_name="0.0.0.0", server_port=2000)

Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


* Running on local URL:  http://0.0.0.0:2000
* To create a public link, set `share=True` in `launch()`.


C:\Users\ameta\anaconda3\Lib\site-packages\gradio\routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Traceback (most recent call last):
  File "C:\Users\ameta\anaconda3\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "C:\Users\ameta\anaconda3\Lib\site-packages\gradio\route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "C:\Users\ameta\anaconda3\Lib\site-packages\gradio\blocks.py", line 2277, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "C:\Users\ameta\anaconda3\Lib\site-p

In [17]:
pip install librosa

  Using cached soundfile-0.14.0-py2.py3-none-win_amd64.whl.metadata (18 kB)
Using cached soundfile-0.14.0-py2.py3-none-win_amd64.whl (1.0 MB)

   -------------------- ------------------- 4/8 [soundfile]
   ------------------------- -------------- 5/8 [pooch]
   ----------------------------------- ---- 7/8 [librosa]
   ----------------------------------- ---- 7/8 [librosa]
   ----------------------------------- ---- 7/8 [librosa]
   ---------------------------------------- 8/8 [librosa]

Note: you may need to restart the kernel to use updated packages.


In [18]:
import torch
from transformers import pipeline
import gradio as gr
import numpy as np
import librosa

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny.en",
    chunk_length_s=30,
)

TARGET_SR = 16000

def transcript_audio(audio_file):
    sampling_rate, audio_array = audio_file

    if audio_array.ndim > 1:
        audio_array = audio_array.mean(axis=1)

    audio_array = audio_array.astype(np.float32)
    max_val = np.abs(audio_array).max()
    if max_val > 0:
        audio_array = audio_array / max_val

    # Manual resample — torchaudio ki koi zaroorat nahi
    if sampling_rate != TARGET_SR:
        audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=TARGET_SR)
        sampling_rate = TARGET_SR

    result = pipe(
        {"array": audio_array, "sampling_rate": sampling_rate},
        batch_size=8
    )["text"]
    return result

audio_input = gr.Audio(sources="upload", type="numpy")
output_text = gr.Textbox()

iface = gr.Interface(
    fn=transcript_audio,
    inputs=audio_input,
    outputs=output_text,
    title="Audio Transcription App",
    description="Upload the audio file"
)

iface.launch(server_name="0.0.0.0", server_port=3000)

Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


* Running on local URL:  http://0.0.0.0:3000
* To create a public link, set `share=True` in `launch()`.


C:\Users\ameta\anaconda3\Lib\site-packages\gradio\routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


In [19]:
import numpy as np
import gradio as gr
from transformers import pipeline
import librosa

from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

# ============================================================
# STEP 1: Whisper — Audio to raw transcript
# ============================================================
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny.en",
    chunk_length_s=30,
)

TARGET_SR = 16000

def transcribe_audio(audio_file):
    sampling_rate, audio_array = audio_file

    if audio_array.ndim > 1:
        audio_array = audio_array.mean(axis=1)

    audio_array = audio_array.astype(np.float32)
    max_val = np.abs(audio_array).max()
    if max_val > 0:
        audio_array = audio_array / max_val

    if sampling_rate != TARGET_SR:
        audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=TARGET_SR)
        sampling_rate = TARGET_SR

    result = asr_pipe({"array": audio_array, "sampling_rate": sampling_rate}, batch_size=8)["text"]
    return result


# ============================================================
# STEP 2: LLAMA 3.2 (Ollama) — Transcript Clean-Up Assistant
# ============================================================
llm = ChatOllama(model="llama3.2", temperature=0.3)

cleanup_prompt = PromptTemplate.from_template("""You are a transcript cleaning assistant. 
You will be given a raw, unedited transcript from a speech-to-text system.

Your task:
- Remove filler words (um, uh, like, you know)
- Fix punctuation and capitalization
- Remove repeated words or stutters
- Keep the original meaning and speaker intent intact
- Do NOT summarize or shorten the content — only clean it up

Raw Transcript:
{raw_transcript}

Cleaned Transcript:""")

cleanup_chain = cleanup_prompt | llm | StrOutputParser()

def clean_transcript(raw_transcript):
    if not raw_transcript or raw_transcript.strip() == "":
        return "No transcript to clean."
    return cleanup_chain.invoke({"raw_transcript": raw_transcript})


# ============================================================
# STEP 3: LLAMA 3.2 (Ollama) — Meeting Minute + Task List Generator
# ============================================================
minutes_prompt = PromptTemplate.from_template("""You are a meeting assistant. 
Based on the cleaned meeting transcript below, generate:

1. A concise summary of the meeting (3-5 bullet points)
2. A list of action items / tasks mentioned, with the responsible person if stated

Format your response exactly like this:

## Meeting Summary
- point 1
- point 2

## Action Items
- [Owner] Task description

Cleaned Transcript:
{cleaned_transcript}
""")

minutes_chain = minutes_prompt | llm | StrOutputParser()

def generate_minutes(cleaned_transcript):
    if not cleaned_transcript or cleaned_transcript.strip() == "":
        return "No transcript to summarize."
    return minutes_chain.invoke({"cleaned_transcript": cleaned_transcript})


# ============================================================
# FULL PIPELINE — Audio -> Raw Transcript -> Clean -> Minutes
# ============================================================
def process_meeting_audio(audio_file):
    if audio_file is None:
        return "No audio provided.", "", ""

    raw_transcript = transcribe_audio(audio_file)
    cleaned_transcript = clean_transcript(raw_transcript)
    meeting_minutes = generate_minutes(cleaned_transcript)

    return raw_transcript, cleaned_transcript, meeting_minutes


# ============================================================
# GRADIO INTERFACE
# ============================================================
audio_input = gr.Audio(sources=["upload", "microphone"], type="numpy", label="Upload / Record Meeting Audio")

raw_output = gr.Textbox(label="Raw Transcript (Whisper)")
cleaned_output = gr.Textbox(label="Cleaned Transcript (LLAMA 3.2)")
minutes_output = gr.Textbox(label="Meeting Minutes + Task List (LLAMA 3.2)")

iface = gr.Interface(
    fn=process_meeting_audio,
    inputs=audio_input,
    outputs=[raw_output, cleaned_output, minutes_output],
    title="AI Meeting Assistant",
    description="Upload or record meeting audio to get transcript, cleaned transcript, and meeting minutes with action items — powered entirely by local Ollama models.",
)

if __name__ == "__main__":
    iface.launch(server_name="0.0.0.0", server_port=4000)

Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


* Running on local URL:  http://0.0.0.0:4000
* To create a public link, set `share=True` in `launch()`.


C:\Users\ameta\anaconda3\Lib\site-packages\gradio\routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
